<a href="https://colab.research.google.com/github/pnhongngoc37-cloud/TH_DeepLearning/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
import os

# ================== KẾT NỐI GOOGLE DRIVE ==================
print("=== KẾT NỐI GOOGLE DRIVE ===")
drive.mount('/content/drive')

# Tạo thư mục làm việc trên Drive
workspace = '/content/drive/My Drive/XetTuyenDH'
os.makedirs(workspace, exist_ok=True)
print(f"Thư mục làm việc: {workspace}")
print()

# ================== TẠO FILE DỮ LIỆU ĐẦU VÀO ==================
print("=== TẠO FILE DULIEUXETTUYENDAIHOC.CSV ===")

np.random.seed(42)  # Cố định kết quả ngẫu nhiên
n_samples = 100

data = {}

# Tạo điểm cho các môn lớp 10, 11, 12
subjects = ['T', 'L', 'H', 'S', 'V', 'X', 'D', 'N']
grades = [1, 2, 6]  # 1: lớp 10, 2: lớp 11, 6: lớp 12

for subject in subjects:
    for grade in grades:
        col_name = f"{subject}{grade}"
        # Điểm từ 3-10, làm tròn 1 chữ số thập phân
        scores = np.random.uniform(3, 10, n_samples)
        scores = np.round(scores, 1)
        # Thêm 5-8% dữ liệu thiếu (NaN)
        missing_idx = np.random.choice(n_samples, size=int(n_samples*np.random.uniform(0.05, 0.08)), replace=False)
        scores[missing_idx] = np.nan
        data[col_name] = scores

# Các biến định tính
data['GT'] = np.random.choice(['Nam', 'Nữ'], n_samples)  # Giới tính
data['DT'] = np.random.choice(['Kinh', 'Tày', 'Thái', 'Mường', np.nan], n_samples,
                               p=[0.7, 0.1, 0.1, 0.05, 0.05])  # Dân tộc
data['KV'] = np.random.choice(['KV1', 'KV2', 'KV3', 'KVNT'], n_samples)  # Khu vực
data['KT'] = np.random.choice(['A', 'A1', 'B', 'C', 'D1'], n_samples)  # Khối thi

# Điểm thi đại học
data['DH1'] = np.round(np.random.uniform(2, 10, n_samples), 1)
data['DH2'] = np.round(np.random.uniform(2, 10, n_samples), 1)
data['DH3'] = np.round(np.random.uniform(2, 10, n_samples), 1)

# Thêm một số missing cho điểm thi
for col in ['DH1', 'DH2', 'DH3']:
    missing_idx = np.random.choice(n_samples, size=int(n_samples*0.03), replace=False)
    data[col][missing_idx] = np.nan

# Tạo DataFrame
df_input = pd.DataFrame(data)

# Đường dẫn file đầu vào trên Drive
input_file_path = os.path.join(workspace, 'dulieuxettuyendaihoc.csv')
df_input.to_csv(input_file_path, index=False, encoding='utf-8-sig')

print(f"Đã tạo file đầu vào tại: {input_file_path}")
print(f"Kích thước: {df_input.shape[0]} dòng, {df_input.shape[1]} cột")
print(f"Số lượng missing ban đầu: {df_input.isnull().sum().sum()}")
print()

# ================== ĐỌC DỮ LIỆU TỪ DRIVE ==================
print("=== ĐỌC DỮ LIỆU TỪ DRIVE ===")
df = pd.read_csv(input_file_path)
print("Đã đọc dữ liệu thành công!")
print(f"10 dòng đầu tiên:")
print(df.head(10))
print(f"\n10 dòng cuối cùng:")
print(df.tail(10))
print()

# ================== 1. PHÂN LOẠI DỮ LIỆU ==================
print("=== 1. Phân loại dữ liệu định tính và định lượng ===")
qualitative_cols = ['GT', 'DT', 'KV', 'KT']
quantitative_cols = ['T1', 'L1', 'H1', 'S1', 'V1', 'X1', 'D1', 'N1',
                     'T2', 'L2', 'H2', 'S2', 'V2', 'X2', 'D2', 'N2',
                     'T6', 'L6', 'H6', 'S6', 'V6', 'X6', 'D6', 'N6',
                     'DH1', 'DH2', 'DH3']
print(f"Biến định tính: {qualitative_cols}")
print(f"Biến định lượng: {quantitative_cols}\n")

# ================== 2. ĐỊNH NGHĨA THANG ĐO ==================
print("=== 2. Định nghĩa thang đo ===")
scales = {
    'GT': 'Nominal (Nhị phân: Nam/Nữ)',
    'DT': 'Nominal (Dân tộc)',
    'KV': 'Nominal (Khu vực ưu tiên)',
    'KT': 'Nominal (Khối thi)',
    'T1..N1, T2..N2, T6..N6': 'Ratio (Điểm từ 0-10)',
    'DH1,DH2,DH3': 'Ratio (Điểm thi ĐH từ 0-10)'
}
for var, scale in scales.items():
    print(f"{var}: {scale}")
print()

# ================== 3. HIỂN THỊ DỮ LIỆU ==================
print("=== 3. Hiển thị dữ liệu (đã làm ở trên) ===")
print()

# ================== 4. XỬ LÝ MISSING CHO DT ==================
print("=== 4. Xử lý dữ liệu thiếu cho cột DT (Dân tộc) ===")
missing_dt = df['DT'].isna().sum()
print(f"Số lượng missing của DT: {missing_dt}")
print(f"Tỷ lệ missing: {missing_dt/len(df)*100:.2f}%")

print("\nTần số các giá trị DT (bao gồm NaN):")
print(df['DT'].value_counts(dropna=False))

unique_dt = df['DT'].unique()
print(f"\nCác giá trị riêng biệt: {unique_dt}")

# Điền giá trị 0 cho missing
df['DT'] = df['DT'].fillna(0)
print("\nSau khi điền 0:")
print(df['DT'].value_counts(dropna=False))
print()

# ================== 5. XỬ LÝ MISSING CHO T1 BẰNG MEAN ==================
print("=== 5. Xử lý dữ liệu thiếu cho T1 bằng Mean ===")
missing_t1 = df['T1'].isna().sum()
print(f"Số lượng missing của T1: {missing_t1}")

print("\nThống kê T1 trước khi xử lý:")
print(df['T1'].describe())

# Tính mean và điền
mean_t1 = df['T1'].mean()
df['T1'] = df['T1'].fillna(mean_t1)
print(f"\nMean của T1: {mean_t1:.2f}")
print("Sau khi điền mean:")
print(df['T1'].describe())
print()

# ================== 6. XỬ LÝ MISSING CHO TẤT CẢ BIẾN ĐIỂM ==================
print("=== 6. Xử lý thiếu cho tất cả biến điểm ===")
score_cols = ['T1', 'L1', 'H1', 'S1', 'V1', 'X1', 'D1', 'N1',
              'T2', 'L2', 'H2', 'S2', 'V2', 'X2', 'D2', 'N2',
              'T6', 'L6', 'H6', 'S6', 'V6', 'X6', 'D6', 'N6',
              'DH1', 'DH2', 'DH3']

for col in score_cols:
    if df[col].isna().sum() > 0:
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val)
        print(f"Đã điền missing cho {col} với mean={mean_val:.2f}")

print(f"\nKiểm tra missing sau khi xử lý: {df[score_cols].isna().sum().sum()} missing còn lại")
print()

# ================== 7. TẠO BIẾN TBM1, TBM2, TBM3 ==================
print("=== 7. Tạo TBM1, TBM2, TBM3 ===")
# TBM = (T*2 + L + H + S + V*2 + X + D + N) / 10
df['TBM1'] = (df['T1']*2 + df['L1'] + df['H1'] + df['S1'] +
              df['V1']*2 + df['X1'] + df['D1'] + df['N1']) / 10
df['TBM2'] = (df['T2']*2 + df['L2'] + df['H2'] + df['S2'] +
              df['V2']*2 + df['X2'] + df['D2'] + df['N2']) / 10
df['TBM3'] = (df['T6']*2 + df['L6'] + df['H6'] + df['S6'] +
              df['V6']*2 + df['X6'] + df['D6'] + df['N6']) / 10
print("Đã tạo TBM1, TBM2, TBM3")
print(df[['TBM1', 'TBM2', 'TBM3']].head())
print()

# ================== 8. TẠO BIẾN XẾP LOẠI ==================
print("=== 8. Tạo xếp loại XL1, XL2, XL3 ===")
def classify_tbm(score):
    if score < 5.0:
        return 'Y'  # Yếu
    elif score < 6.5:
        return 'TB'  # Trung bình
    elif score < 8.0:
        return 'K'   # Khá
    elif score < 9.0:
        return 'G'   # Giỏi
    else:
        return 'XS'  # Xuất sắc

df['XL1'] = df['TBM1'].apply(classify_tbm)
df['XL2'] = df['TBM2'].apply(classify_tbm)
df['XL3'] = df['TBM3'].apply(classify_tbm)
print("Đã tạo XL1, XL2, XL3")
print(df[['TBM1', 'XL1', 'TBM2', 'XL2', 'TBM3', 'XL3']].head())
print()

# ================== 9. MIN-MAX NORMALIZATION ==================
print("=== 9. Min-Max Normalization sang thang điểm 4 ===")
def min_max_to_4(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([2.0] * len(series))
    return ((series - min_val) / (max_val - min_val)) * 4

df['US_TBM1'] = min_max_to_4(df['TBM1'])
df['US_TBM2'] = min_max_to_4(df['TBM2'])
df['US_TBM3'] = min_max_to_4(df['TBM3'])
print("Đã tạo US_TBM1, US_TBM2, US_TBM3")
print(df[['TBM1', 'US_TBM1', 'TBM2', 'US_TBM2', 'TBM3', 'US_TBM3']].head())
print()

# ================== 10. TẠO BIẾN KẾT QUẢ XÉT TUYỂN ==================
print("=== 10. Tạo biến KQXT (Đậu/Rớt) ===")
def calculate_kqxt(row):
    khoi = row['KT']
    dh1, dh2, dh3 = row['DH1'], row['DH2'], row['DH3']

    if khoi in ['A', 'A1']:
        score = (dh1*2 + dh2 + dh3) / 4
    elif khoi == 'B':
        score = (dh1 + dh2*2 + dh3) / 4
    else:
        score = (dh1 + dh2 + dh3) / 3

    return 1 if score >= 5.0 else 0

df['KQXT'] = df.apply(calculate_kqxt, axis=1)
print("Đã tạo KQXT")
print(f"Số lượng đậu (1): {df['KQXT'].sum()}")
print(f"Số lượng rớt (0): {len(df) - df['KQXT'].sum()}")
print(df[['KT', 'DH1', 'DH2', 'DH3', 'KQXT']].head())
print()

# ================== 11. LƯU DỮ LIỆU VÀO DRIVE ==================
print("=== 11. Lưu dữ liệu đã xử lý vào Drive ===")
output_file_path = os.path.join(workspace, 'processed_dulieuxettuyendaihoc.csv')
df.to_csv(output_file_path, index=False, encoding='utf-8-sig')
print(f"Đã lưu dữ liệu vào: {output_file_path}")
print(f"Kích thước dữ liệu cuối cùng: {df.shape}")
print(f"Các cột mới được thêm: TBM1, TBM2, TBM3, XL1, XL2, XL3, US_TBM1, US_TBM2, US_TBM3, KQXT")

# ================== 12. HIỂN THỊ THÔNG TIN TỔNG KẾT ==================
print("\n" + "="*50)
print("=== TỔNG KẾT ===")
print("="*50)
print(f"Thư mục làm việc trên Drive: {workspace}")
print(f"File đầu vào: dulieuxettuyendaihoc.csv")
print(f"File đầu ra: processed_dulieuxettuyendaihoc.csv")
print(f"Số dòng: {len(df)}")
print(f"Tổng số cột: {len(df.columns)}")
print(f"Số cột mới tạo: 10 cột")

# Hiển thị đường dẫn đầy đủ
print(f"\nĐường dẫn đầy đủ đến file đã xử lý:")
print(f"{output_file_path}")

# Tùy chọn tải file về máy
download = input("\nBạn có muốn tải file đã xử lý về máy tính? (y/n): ")
if download.lower() == 'y':
    from google.colab import files
    files.download(output_file_path)
    print("Đã tải file về máy tính!")